# DALES SuperPoint Transformer - Visualization
Load trained model and visualize predictions without ConfigUI

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import torch
import hydra

# Add project to path
sys.path.insert(0, '/cluster/home/larshfle/superpoint_transformer_new')

from src.utils import init_config
from src.transforms import *
from src.data import *

print("✅ Imports successful!")

✅ Imports successful!


## 1. Setup paths and config

In [2]:
# Settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
split = 'test'  # Choose: 'train', 'val', or 'test'
ckpt_path = '/cluster/home/larshfle/superpoint_transformer_new/logs/train/runs/2026-03-13_03-10-14/checkpoints/epoch_399.ckpt'
data_dir = '/cluster/home/larshfle/datasets/dales_ply'

print(f"Device: {device}")
print(f"Split: {split}")
print(f"Checkpoint: {ckpt_path}")
print(f"Data directory: {data_dir}")

# Verify checkpoint exists
assert os.path.exists(ckpt_path), f"Checkpoint not found at {ckpt_path}"
print("✅ Checkpoint found!")

Device: cuda
Split: test
Checkpoint: /cluster/home/larshfle/superpoint_transformer_new/logs/train/runs/2026-03-13_03-10-14/checkpoints/epoch_399.ckpt
Data directory: /cluster/home/larshfle/datasets/dales_ply
✅ Checkpoint found!


## 2. Load config and instantiate datamodule

In [4]:
# Parse config with overrides
cfg = init_config(overrides=[
    'experiment=semantic/dales',
    f'ckpt_path={ckpt_path}',
    'datamodule.mini=false',
    'datamodule.load_full_res_idx=true',
])

# Override data_dir if needed
cfg.datamodule.data_dir = data_dir

print("✅ Config loaded!")
print(f"Experiment: {cfg.get('experiment', 'semantic/dales')}")
print(f"Model: {cfg.model._target_}")

✅ Config loaded!
Experiment: semantic/dales
Model: src.models.semantic.SemanticSegmentationModule


In [5]:
# Instantiate datamodule
datamodule = hydra.utils.instantiate(cfg.datamodule)
datamodule.prepare_data()
datamodule.setup()

# Get dataset
if split == 'train':
    dataset = datamodule.train_dataset
elif split == 'val':
    dataset = datamodule.val_dataset
elif split == 'test':
    dataset = datamodule.test_dataset
else:
    raise ValueError(f"Unknown split '{split}'")

print(f"✅ Datamodule loaded! Split: {split}")
dataset.print_classes()

✅ Datamodule loaded! Split: test
0   Ground               stuff
1   Not Ground           stuff
2   Ignored              void


## 3. Load model

In [ ]:
# Instantiate model
model = hydra.utils.instantiate(cfg.model)

# Load checkpoint
load_kwargs = {}
pretrained_cnn_ckpt_path = cfg.datamodule.get("pretrained_cnn_ckpt_path", None)
if pretrained_cnn_ckpt_path is not None:
    load_kwargs["pretrained_cnn_ckpt_path"] = pretrained_cnn_ckpt_path

model = model._load_from_checkpoint(
    cfg.ckpt_path,
    **load_kwargs
)

# Move to device and eval mode
model = model.eval().to(device)

print(f"✅ Model loaded from checkpoint!")
print(f"Device: {device}")

ConfigAttributeError: Key 'pretrained_cnn_ckpt_path' is not in struct
    full_key: datamodule.pretrained_cnn_ckpt_path
    object_type=dict

## 4. Run inference on first sample

In [ ]:
# Enable feature storage for visualization
model.net.store_features = True

# Load first sample
print("Loading first sample...")
nag = dataset[0]

# Apply on-device transforms
nag = dataset.on_device_transform(nag.to(device))

print(f"✅ Sample loaded!")
print(f"  - Level 0 points: {nag[0].pos.shape[0]}")
print(f"  - Level 1 superpoints: {nag[1].pos.shape[0]}")

In [ ]:
# Run inference
print("Running inference...")
with torch.no_grad():
    output = model(nag)

print("✅ Inference complete!")

# Get voxel-wise predictions
nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)

# Get object/panoptic predictions if applicable
if hasattr(output, 'voxel_panoptic_pred'):
    vox_y, vox_index, vox_obj_pred = output.voxel_panoptic_pred(super_index=nag[0].super_index)
    nag[0].obj_pred = vox_obj_pred
    print("  - Panoptic predictions added")
    
print(f"Predictions shape: {nag[0].semantic_pred.shape}")

## 5. Visualize full tile

In [ ]:
# Visualize hierarchical partition
nag.show(
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000
)

## 6. Visualize a cropped region

In [ ]:
# Define region
center = nag[0].pos.mean(dim=0).view(1, -1)
radius = 10  # DALES is typically in meters

print(f"Center: {center.squeeze().tolist()}")
print(f"Radius: {radius}")

# Visualize
nag.show(
    radius=radius,
    center=center,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000
)

## 7. Visualize graph structure

In [ ]:
# Visualize with superpoint centroids and edges
nag.show(
    radius=radius,
    center=center,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000,
    centroids=True,
    h_edge=True,
    h_edge_width=2
)

## 8. Export to HTML

In [ ]:
# Export visualization
output_path = '/cluster/home/larshfle/superpoint_transformer_new/dales_visualization.html'

nag.show(
    figsize=1600,
    radius=radius,
    center=center,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000,
    title="DALES SPT Predictions (epoch_399)",
    path=output_path
)

print(f"✅ Visualization exported to: {output_path}")